# Estudo do modelo: Sonic Sleuth

- **Família:** Espectrograma (features in-model)
- **Tipo de entrada (registry):** `spectrogram`
- **Por quê:** Extrai LFCC/MFCC/CQT dentro do grafo (tf.signal).

## Objetivos

1. Mostrar o **contrato de entrada** que o benchmark prepara para esta
   arquitetura.
2. Gerar dados sintéticos mínimos para validar shape e fluxo.
3. Instanciar/rodar o modelo pelo **mesmo caminho** do benchmark.

Para treino real e métricas, use `notebooks/pipeline/`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "app").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT)

In [ ]:
ARCHITECTURE = 'Sonic Sleuth'

from benchmarks.data import BenchmarkData

data = BenchmarkData.synthetic(n=48, shape=(64, 40), seed=7)
prepared = data.prepare_for_architecture(ARCHITECTURE)
print("Arquitetura     :", ARCHITECTURE)
print("Shape original  :", data.X.shape)
print("Shape preparado :", prepared.X.shape)
print("Metadados       :", prepared.metadata)

## Inspeção do modelo

In [ ]:
ARCHITECTURE = 'Sonic Sleuth'

# Modelos neurais são instanciados pelo factory (mesmo caminho do
# benchmark/treino). summary() valida shapes sem treino pesado.
from app.domain.models.architectures.factory import create_model_by_name

input_shape = tuple(prepared.X.shape[1:])
model = create_model_by_name(ARCHITECTURE, input_shape=input_shape, num_classes=1)
model.summary()
print("Parâmetros:", model.count_params())

## Como ler este notebook

- `docs/08_ARQUITETURAS.md` — descrição de todas as arquiteturas.
- `docs/10_TREINAMENTO.md` — hiperparâmetros e estratégia de treino.
- `docs/15_BENCHMARK.md` — execução completa, métricas e gráficos.

No relatório, compare sempre: acurácia, EER, AUC-ROC, min-tDCF,
latência, tamanho do modelo, matriz de confusão e distribuição de scores.

## Hiperparâmetros e análise esperada

- **Treino rápido:** use `epochs=2` apenas para validar fluxo.
- **Treino para TCC:** use o preset do benchmark completo em
  `notebooks/pipeline/01_benchmark_tcc_full_pipeline.ipynb`.
- **Entrada preparada:** confira a célula de `BenchmarkData` acima;
  ela mostra o shape real usado no harness.
- **Arquivos esperados no relatório:** `architectures/sonic_sleuth/metrics.json`,
  `confusion_matrix.png`, `roc.png`, `score_distribution.png` e,
  quando houver histórico, `convergence.png`.

In [ ]:
import json

metrics_path = ROOT / "results" / "tcc_full_20k" / "architectures" / 'sonic_sleuth' / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print(json.dumps(metrics.get("clean", metrics), indent=2, ensure_ascii=False)[:2000])
else:
    print("Sem métricas salvas ainda:", metrics_path)
    print("Rode o benchmark completo ou ajuste o caminho de results.")